## String and comparison evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/pl/6_2_Evaluation_string_and_comparison.ipynb)

| | |
|---|---|
| 🎯 **Czego się nauczysz** | Porównywanie odpowiedzi za pomocą odległości embeddingów, odległości tekstowej i oceny parami przez LLM. |
| 🔑 **Klucze API** | `OPENAI_API_KEY` |
| 💲 **Koszt API** | bardzo niski – kilka krótkich zapytań |
| ⏱️ **Czas wykonania** | < 30 s |

In [ ]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
# Pomijane, gdy brak pip, np. w środowisku `uv sync`, które ma je już zainstalowane (zob. README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU langchain-classic langchain-openai sacrebleu rouge-score nltk python-dotenv rapidfuzz
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.

In [2]:
import json
from dotenv import load_dotenv

load_dotenv()

True

### Embedding Distance Evaluator
Embedding Distance Evaluator porównuje dwie odpowiedzi, zamieniając je na wektory osadzeń (embeddings) i mierząc odległość lub podobieństwo kosinusowe. Dzięki temu ocenia semantyczną bliskość treści, a nie tylko dopasowanie słów.

In [3]:
from langchain_classic.evaluation import load_evaluator

evaluator = load_evaluator("embedding_distance", embeddings_model="openai")

result1 = evaluator.evaluate_strings(
    prediction="Stolica Polski to Warszawa",
    reference="Stolica Polski to Warszawa"
)

result2 = evaluator.evaluate_strings(
    prediction="Stolica Polski to Warszawa",
    reference="Stolica Polski nosi nazwę Warszawa"
)

result3 = evaluator.evaluate_strings(
    prediction="Stolica Polski to Warszawa",
    reference="Stolica Burkina Faso nosi nazwę Wagadugu"
)

print(round(result1["score"], 4))
print(round(result2["score"], 4))
print(round(result3["score"], 4))


-0.0
0.0508
0.1318


### String Comparison
Ewaluator porównuje dwa teksty przy użyciu metryki BLEU, która mierzy n-gramowe podobieństwo wygenerowanej odpowiedzi do odpowiedzi referencyjnej.

In [4]:
evaluator = load_evaluator("string_distance")

result1 = evaluator.evaluate_strings(
    prediction="Stolica Polski to Warszawa",
    reference="Stolica Polski to Warszawa"
)

result2 = evaluator.evaluate_strings(
    prediction="Stolicą Polski jest Warszawa",
    reference="Stolica Polski to Warszawa"
)

result3 = evaluator.evaluate_strings(
    prediction="Stolica Polski to Warszawa",
    reference="Warszawa jest stolicą Polski"
)

print(round(result1["score"], 4))
print(round(result2["score"], 4))
print(round(result3["score"], 4))


0.0
0.069
0.4991


### Testy A/B
PairwiseStringEvaluator służy do porównywania dwóch odpowiedzi tekstowych względem jednej referencji, aby wybrać lepszą. Dzięki temu można automatycznie ocenić, która z odpowiedzi jest bliższa oczekiwanemu wynikowi.

In [5]:
from langchain_classic.evaluation import load_evaluator

evaluator = load_evaluator("labeled_pairwise_string")

result = evaluator.evaluate_string_pairs(
    input="What is the capital of Poland?",
    prediction="Warsaw is the capital of Poland",
    prediction_b="I don't know",
    reference="Warsaw is Poland's capital"
)

print(json.dumps(result, indent=4))

{
    "reasoning": "Assistant A's response is helpful, relevant, correct, and accurate. It directly answers the user's question about the capital of Poland. On the other hand, Assistant B's response is not helpful or accurate. It does not provide the user with the information they were seeking. Therefore, Assistant A's response is superior in this case. \n\nFinal verdict: [[A]]",
    "value": "A",
    "score": 1
}
